In [0]:
# ── Kafka / Confluent Cloud connection details ────────────────────────────────
KAFKA_BOOTSTRAP_SERVERS = dbutils.secrets.get(scope="finguard secret vault", key="kafka_bootstrap_servers")
KAFKA_TOPIC            = dbutils.secrets.get(scope="finguard secret vault", key="kafka_topic")
API_KEY                = dbutils.secrets.get(scope="finguard secret vault", key="api_key")
API_SECRET             = dbutils.secrets.get(scope="finguard secret vault", key="api_secret")

JAAS_CONFIG = (
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="{API_KEY}" password="{API_SECRET}";'
)

# ── Batch read – all messages from earliest to latest offset ──────────────────
kafka_batch_df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers",   KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe",                  KAFKA_TOPIC)
    .option("startingOffsets",            "earliest")
    .option("kafka.security.protocol",   "SASL_SSL")
    .option("kafka.sasl.mechanism",      "PLAIN")
    .option("kafka.sasl.jaas.config",    JAAS_CONFIG)
    .load()
)


In [0]:
kafka_batch_df.count()

In [0]:
display(kafka_batch_df.limit(10))

In [0]:
from pyspark.sql.functions import col

# Cast binary columns (key, value) to string; keep other columns as-is
kafka_parsed_df = kafka_batch_df.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    "topic",
    "partition",
    "offset",
    "timestamp",
    "timestampType"
)

display(kafka_parsed_df.limit(5))

In [0]:
kafka_parsed_df.write.mode("overwrite").saveAsTable("finguard.bronze_kafka.transactions_batch")

In [0]:
# ── Streaing read – 
kafka_streaming_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers",   KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe",                  KAFKA_TOPIC)
    .option("startingOffsets",            "earliest")
    .option("kafka.security.protocol",   "SASL_SSL")
    .option("kafka.sasl.mechanism",      "PLAIN")
    .option("kafka.sasl.jaas.config",    JAAS_CONFIG)
    .load()
)

In [0]:
display(kafka_streaming_df)

In [0]:
from pyspark.sql.functions import col

# Cast binary columns (key, value) to string; keep other columns as-is
kafka_streaming_parsed_df = kafka_streaming_df.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    "topic",
    "partition",
    "offset",
    "timestamp",
    "timestampType"
)


In [0]:
display(kafka_streaming_parsed_df.limit(5))

In [0]:
streaming_query=(kafka_streaming_parsed_df.writeStream
                 .format("delta").outputMode("append")
                 .option("checkpointLocation", "/Volumes/finguard/bronze_blob/transactions_volume/_checkpoint/")
                 .trigger(availableNow=True)
                 .toTable("finguard.bronze_kafka.transactions_streaming")
                            
)
print("Streaming query started, query id is : ",streaming_query.id)